# Assignment 1

bfs dfs

In [7]:
%%writefile ass1.cu
#include<iostream>
#include<omp.h>
#include<bits/stdc++.h>

using namespace std;


class Graph{
    public:
        // vector<vector<int>> graph;
        // vector<bool> visited;
        // int vertices = 0;
        // int edges = 0;

        int vertices = 6;
        int edges = 5;
        vector<vector<int>> graph = {{1},{0,2,3},{1,4,5},{1,4},{2,3},{2}};
        vector<bool> visited;

        // Graph(){
        //     cout << "Enter number of nodes: ";
        //     cin >> vertices;
        //     cout << "Enter number of edges: ";
        //     cin >> edges;
        //     graph.assign(vertices,vector<int>());
        //     for(int i = 0 ; i < edges;i++){
        //         int a,b;
        //         cout << "Enter adjacent nodes: ";
        //         cin >> a >> b;
        //         addEdge(a,b);
        //     }
        // }
        void addEdge(int a, int b){
            graph[a].push_back(b);
            graph[b].push_back(a);
        }

        void printGraph(){
            for(int i = 0; i < vertices; i++){
                cout << i << " -> ";
                for(auto j = graph[i].begin(); j != graph[i].end();j++){
                    cout << *j << " ";
                }
                cout << endl;
            }
        }

        void initialize_visited(){
            visited.assign(vertices,false);
        }

        void dfs(int i){
            stack<int> s;
            s.push(i);
            visited[i] = true;

            while(s.empty() != true){
                int current = s.top();
                cout << current << " ";
                s.pop();
                for(auto j = graph[current].begin(); j != graph[current].end();j++){
                    if(visited[*j] == false){
                        s.push(*j);
                        visited[*j] = true;
                    }
                }

            }
        }

        void parallel_dfs(int i){
            stack<int> s;
            s.push(i);
            visited[i] = true;

            while(s.empty() != true){
                int current = s.top();
                cout << current << " ";
                #pragma omp critical
                    s.pop();
                #pragma omp parallel for
                    for(auto j = graph[current].begin(); j != graph[current].end();j++){
                        if(visited[*j] == false){
                            #pragma omp critical
                            {
                                s.push(*j);
                                visited[*j] = true;
                            }
                        }
                    }

            }
        }

        void bfs(int i){
            queue<int> q;
            q.push(i);
            visited[i] = true;

            while(q.empty() != true){
                int current = q.front();
                q.pop();
                cout << current << " ";
                for(auto j = graph[current].begin(); j != graph[current].end();j++){
                    if(visited[*j] == false){
                        q.push(*j);
                        visited[*j] = true;
                    }
                }
            }
        }

        void parallel_bfs(int i){
            queue<int> q;
            q.push(i);
            visited[i] = true;

            while(q.empty() != true){

                    int current = q.front();
                    cout << current << " ";
                    #pragma omp critical
                        q.pop();

                #pragma omp parallel for
                    for(auto j = graph[current].begin(); j != graph[current].end();j++){
                        if(visited[*j] == false){
                            #pragma omp critical
                                q.push(*j);
                                visited[*j] = true;
                        }
                    }

            }
        }
};

int main(int argc, char const *argv[])
{
    Graph g;
    cout << "Adjacency List:\n";
    g.printGraph();
    g.initialize_visited();
    cout << "Depth First Search: \n";
    auto start = chrono::high_resolution_clock::now();
    g.dfs(0);
    cout << endl;
    auto end = chrono::high_resolution_clock::now();
    cout << "Time taken: " << chrono::duration_cast<chrono::microseconds>(end - start).count() << " microseconds" << endl;
    cout << "Parallel Depth First Search: \n";
    g.initialize_visited();
    start = chrono::high_resolution_clock::now();
    g.parallel_dfs(0);
    cout << endl;
    end = chrono::high_resolution_clock::now();
    cout << "Time taken: "<< chrono::duration_cast<chrono::microseconds>(end - start).count() << " microseconds" << endl;
    start = chrono::high_resolution_clock::now();
    cout << "Breadth First Search: \n";
    g.initialize_visited();
    g.bfs(0);
    cout << endl;
    end = chrono::high_resolution_clock::now();
    cout << "Time taken: "<< chrono::duration_cast<chrono::microseconds>(end - start).count() << " microseconds" << endl;
    start = chrono::high_resolution_clock::now();
    cout << "Parallel Breadth First Search: \n";
    g.initialize_visited();
    g.parallel_bfs(0);
    cout << endl;
    end = chrono::high_resolution_clock::now();
    cout << "Time taken: " << chrono::duration_cast<chrono::microseconds>(end - start).count() << " microseconds" << endl;

    return 0;
}


Overwriting ass1.cu


In [8]:
!nvcc -Xcompiler -fopenmp ass1.cu -o ass1
!./ass1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Adjacency List:
0 -> 1 
1 -> 0 2 3 
2 -> 1 4 5 
3 -> 1 4 
4 -> 2 3 
5 -> 2 
Depth First Search: 
0 1 3 4 2 5 
Time taken: 8 microseconds
Parallel Depth First Search: 
0 1 3 4 2 5 
Time taken: 80 microseconds
Breadth First Search: 
0 1 2 3 4 5 
Time taken: 5 microseconds
Parallel Breadth First Search: 
0 1 3 2 4 5 
Time taken: 8 microseconds


# Assignment 2

Sequential Bubble Sort
Parallel Bubble Sort
Sequential Merge Sort
Parallel Merge Sort

In [ ]:
%%writefile ass2.cu
#include <iostream>
#include <omp.h>
#include <vector>
#include <cstdlib>

using namespace std;

void sequential_bubble_sort(vector<int> arr) {
    double start = omp_get_wtime();

    int size = arr.size();
    for(int i = 0; i < size - 1; i++){
        for(int j = 0; j < size - i - 1; j++){
            if(arr[j] > arr[j+1]){
                swap(arr[j], arr[j+1]);
            }
        }
    }

    double end = omp_get_wtime();
    cout << "Sequential Bubble Sort Time: " << end - start << endl;
}

void parallel_bubble_sort(vector<int> arr) {
    int size = arr.size();

    double start = omp_get_wtime();

    for(int k = 0; k < size; k++){
        if(k % 2 == 0){
            #pragma omp parallel for
            for(int i = 1; i < size - 1; i += 2){
                if(arr[i] > arr[i+1]){
                    swap(arr[i], arr[i+1]);
                }
            }
        } else {
            #pragma omp parallel for
            for(int i = 0; i < size - 1; i += 2){
                if(arr[i] > arr[i+1]){
                    swap(arr[i], arr[i+1]);
                }
            }
        }
    }

    double end = omp_get_wtime();
    cout << "Parallel Bubble Sort Time: " << end - start << endl;
}

void merge(vector<int>& arr, int low, int mid, int high) {
    vector<int> temp;
    int i = low, j = mid + 1;

    while(i <= mid && j <= high){
        if(arr[i] <= arr[j]){
            temp.push_back(arr[i++]);
        } else {
            temp.push_back(arr[j++]);
        }
    }

    while(i <= mid) temp.push_back(arr[i++]);
    while(j <= high) temp.push_back(arr[j++]);

    for(int k = 0; k < temp.size(); k++){
        arr[low + k] = temp[k];
    }
}

void mergesort(vector<int>& arr, int low, int high) {
    if(low < high){
        int mid = (low + high) / 2;
        mergesort(arr, low, mid);
        mergesort(arr, mid+1, high);
        merge(arr, low, mid, high);
    }
}

void parallel_mergesort(vector<int>& arr, int low, int high) {
    if(low < high){
        int mid = (low + high) / 2;

        #pragma omp parallel sections
        {
            #pragma omp section
            parallel_mergesort(arr, low, mid);

            #pragma omp section
            parallel_mergesort(arr, mid+1, high);
        }

        merge(arr, low, mid, high);
    }
}

int main() {
    int SIZE;
    cout << "Enter size of array: ";
    cin >> SIZE;

    vector<int> arr(SIZE);
    for(int i = 0; i < SIZE; i++){
        arr[i] = rand() % 1000;
    }

    sequential_bubble_sort(arr);
    parallel_bubble_sort(arr);

    vector<int> arr_copy1 = arr;
    vector<int> arr_copy2 = arr;

    double start = omp_get_wtime();
    mergesort(arr_copy1, 0, SIZE-1);
    double end = omp_get_wtime();
    cout << "Merge Sort Time: " << end - start << endl;

    start = omp_get_wtime();
    parallel_mergesort(arr_copy2, 0, SIZE-1);
    end = omp_get_wtime();
    cout << "Parallel Merge Sort Time: " << end - start << endl;

    return 0;
}

Writing ass2.cu


In [ ]:
!nvcc -Xcompiler -fopenmp ass2.cu -o ass2
!./ass2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter size of array: 6
Sequential Bubble Sort Time: 8.67e-07
Parallel Bubble Sort Time: 9.326e-05
Merge Sort Time: 4.889e-06
Parallel Merge Sort Time: 4.2357e-05


# Assignment 3

array operatios = min max sum average

In [ ]:
%%writefile ass3.cu
#include<iostream>
#include<omp.h>
#include<bits/stdc++.h>

using namespace std;

void minimum(vector<int> array){
    int min = INT_MAX;
    double start = omp_get_wtime();
    for(auto i = array.begin(); i != array.end();i++){
        if(*i < min){
            min = *i;
        }
    }
    double end = omp_get_wtime();
    cout << "Minimum Element: " << min << endl;
    cout << "Time Taken: " << (end-start) << endl;
    int min_ele = INT_MAX;
    start = omp_get_wtime();
    #pragma omp parallel for reduction(min: min_ele)
        for(auto i = array.begin(); i != array.end();i++){
            if(*i < min_ele){
                min_ele = *i;
            }
        }
    end = omp_get_wtime();
    cout << "Minimum Element(Parallel Reduction): " << min_ele << endl;
    cout << "Time Taken: " << (end-start) << endl;

}

void maximum(vector<int> array){
    int max = INT_MIN;
    double start = omp_get_wtime();
    for(auto i = array.begin(); i != array.end();i++){
        if(*i > max){
            max = *i;
        }
    }
    double end = omp_get_wtime();
    cout << "Maximum Element: " << max << endl;
    cout << "Time Taken: " << (end-start) << endl;
    int max_ele = INT_MIN;
    start = omp_get_wtime();
    #pragma omp parallel for reduction(max: max_ele)
        for(auto i = array.begin(); i != array.end();i++){
            if(*i > max_ele){
                max_ele = *i;
            }
        }
    end = omp_get_wtime();
    cout << "Maximum Element(Parallel Reduction): " << max_ele << endl;
    cout << "Time Taken: " << (end-start) << endl;

}

void sum(vector<int> array){
    int sum = 0;
    double start = omp_get_wtime();
    for(auto i = array.begin(); i != array.end();i++){
        sum += *i;
    }
    double end = omp_get_wtime();
    cout << "Summation: " << sum << endl;
    cout << "Time Taken: " << (end-start) << endl;
    sum = 0;
    start = omp_get_wtime();
    #pragma omp parallel for reduction(+: sum)
        for(auto i = array.begin(); i != array.end();i++){
            sum += *i;
        }
    end = omp_get_wtime();
    cout << "Summation(Parallel Reduction): " << sum << endl;
    cout << "Time Taken: " << (end-start) << endl;

}
void average(vector<int> array){
    float avg = 0;
    double start = omp_get_wtime();
    for(auto i = array.begin(); i != array.end();i++){
        avg += *i;
    }
    double end = omp_get_wtime();
    cout << "Average: " << avg / array.size() << endl;
    cout << "Time Taken: " << (end-start) << endl;
    avg = 0;
    start = omp_get_wtime();
    #pragma omp parallel for reduction(+: avg)
        for(auto i = array.begin(); i != array.end();i++){
            avg += *i;
        }
    end = omp_get_wtime();
    cout << "Average(Parallel Reduction): " << avg / array.size() << endl;
    cout << "Time Taken: " << (end-start) << endl;

}

int main(){
    cout << "Enter number of elements in array: ";
    int N;
    int MAX = 1000;
    cin >> N;
    vector<int> array;
    for(int i = 0; i < N; i++){
        array.push_back(rand() % MAX);
    }
    minimum(array);
    maximum(array);
    sum(array);
    average(array);
    return 0;
}

Overwriting ass3.cu


In [ ]:
!nvcc -Xcompiler -fopenmp ass3.cu -o ass3
!./ass3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter number of elements in array: 7
Minimum Element: 335
Time Taken: 8.54e-07
Minimum Element(Parallel Reduction): 335
Time Taken: 8.6732e-05
Maximum Element: 915
Time Taken: 4.25e-07
Maximum Element(Parallel Reduction): 915
Time Taken: 1.512e-06
Summation: 4475
Time Taken: 3.28e-07
Summation(Parallel Reduction): 4475
Time Taken: 9.76e-07
Average: 639.286
Time Taken: 3.23e-07
Average(Parallel Reduction): 639.286
Time Taken: 9.49e-07


# Assignment 4 : cuda

## 1) Checked GPU **availability**

In [ ]:
!nvidia-smi

Wed Apr 22 17:29:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!/usr/local/cuda/bin/nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


# 4 A) vector addition

In [ ]:
%%writefile vector.cu
#include <iostream>
#include <cuda_runtime.h>
using namespace std;

__global__ void addVectors(int* A, int* B, int* C, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        C[i] = A[i] + B[i];
}

int main()
{
    int n = 1000000;
    int size = n * sizeof(int);

    int *A, *B, *C;
    cudaMallocHost(&A, size);
    cudaMallocHost(&B, size);
    cudaMallocHost(&C, size);

    for (int i = 0; i < n; i++)
    {
        A[i] = i;
        B[i] = i * 2;
    }

    int *dev_A, *dev_B, *dev_C;
    cudaMalloc(&dev_A, size);
    cudaMalloc(&dev_B, size);
    cudaMalloc(&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    int blockSize = 256;
    int numBlocks = (n + blockSize - 1) / blockSize;

    addVectors<<<numBlocks, blockSize>>>(dev_A, dev_B, dev_C, n);

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    for (int i = 0; i < 10; i++)
        cout << C[i] << " ";

    cout << endl;

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
    cudaFreeHost(A);
    cudaFreeHost(B);
    cudaFreeHost(C);

    return 0;
}

Writing vector.cu


In [ ]:
!nvcc vector.cu -o vector

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vector

0 3 6 9 12 15 18 21 24 27 


# 4 B) Matrix Multiplication

In [ ]:
%%writefile matmul.cu
#include <cuda_runtime.h>
#include <iostream>
using namespace std;

__global__ void matmul(int* A, int* B, int* C, int N)
{
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if (Row < N && Col < N)
    {
        int Pvalue = 0;

        for (int k = 0; k < N; k++)
        {
            Pvalue += A[Row * N + k] * B[k * N + Col];
        }

        C[Row * N + Col] = Pvalue;
    }
}

int main()
{
    int N = 512;
    int size = N * N * sizeof(int);

    int *A, *B, *C;
    int *dev_A, *dev_B, *dev_C;

    // Allocate host memory
    cudaMallocHost(&A, size);
    cudaMallocHost(&B, size);
    cudaMallocHost(&C, size);

    // Allocate device memory
    cudaMalloc(&dev_A, size);
    cudaMalloc(&dev_B, size);
    cudaMalloc(&dev_C, size);

    // Initialize matrices
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            A[i * N + j] = i * N + j;
            B[i * N + j] = j * N + i;
        }
    }

    // Copy to GPU
    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    // Define block and grid size
    dim3 dimBlock(16, 16);
    dim3 dimGrid((N + dimBlock.x - 1) / dimBlock.x,
                 (N + dimBlock.y - 1) / dimBlock.y);

    // Launch kernel
    matmul<<<dimGrid, dimBlock>>>(dev_A, dev_B, dev_C, N);

    // Copy result back
    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    // Print first 10x10 result matrix
    for (int i = 0; i < 10; i++)
    {
        for (int j = 0; j < 10; j++)
        {
            cout << C[i * N + j] << " ";
        }
        cout << endl;
    }

    // Free memory
    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    cudaFreeHost(A);
    cudaFreeHost(B);
    cudaFreeHost(C);

    return 0;
}

Overwriting matmul.cu


In [ ]:
!nvcc matmul.cu -o matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matmul

44608256 111586048 178563840 245541632 312519424 379497216 446475008 513452800 580430592 647408384 
111586048 312781568 513977088 715172608 916368128 1117563648 1318759168 1519954688 1721150208 1922345728 
178563840 513977088 849390336 1184803584 1520216832 1855630080 -2103923968 -1768510720 -1433097472 -1097684224 
245541632 715172608 1184803584 1654434560 2124065536 -1701270784 -1231639808 -762008832 -292377856 177253120 
312519424 916368128 1520216832 2124065536 -1567053056 -963204352 -359355648 244493056 848341760 1452190464 
379497216 1117563648 1855630080 -1701270784 -963204352 -225137920 512928512 1250994944 1989061376 -1567839488 
446475008 1318759168 -2103923968 -1231639808 -359355648 512928512 1385212672 -2037470464 -1165186304 -292902144 
513452800 1519954688 -1768510720 -762008832 244493056 1250994944 -2037470464 -1030968576 -24466688 982035200 
580430592 1721150208 -1433097472 -292377856 848341760 1989061376 -1165186304 -24466688 1116252928 -2037994752 
647408384 192234572

# Mini Project

##  Mini Project: Implement Non-Serial Polyadic Dynamic Programming with GPU Parallelization

Find minimum number of scalar multiplications needed to multiply matrices

Input (matrix sizes)
↓
Initialize DP table
↓
GPU computes diagonals in parallel
↓
Each thread calculates min cost for (i,j)
↓
Final result copied back to CPU
↓
Print minimum cost

In [ ]:
%%writefile mcm.cu
#include <iostream>
#include <climits>
#include <cuda_runtime.h>
using namespace std;

__global__ void computeDiagonal(int *dp, int *p, int n, int len)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    int j = i + len - 1;

    if (j < n)
    {
        int best = INT_MAX;

        for (int k = i; k < j; k++)
        {
            int cost = dp[i*n + k] +
                       dp[(k+1)*n + j] +
                       p[i-1] * p[k] * p[j];

            if (cost < best) best = cost;
        }

        dp[i*n + j] = best;
    }
}

int main()
{
    int p[] = {10,20,30,40,30};
    int n = 5;

    int sizeDP = n * n * sizeof(int);
    int sizeP = n * sizeof(int);

    int *dp = new int[n*n]();
    int *d_dp, *d_p;

    cudaMalloc(&d_dp, sizeDP);
    cudaMalloc(&d_p, sizeP);

    cudaMemcpy(d_dp, dp, sizeDP, cudaMemcpyHostToDevice);
    cudaMemcpy(d_p, p, sizeP, cudaMemcpyHostToDevice);

    for(int len=2; len<n; len++)
    {
        int blocks = 1;
        int threads = 256;
        computeDiagonal<<<blocks,threads>>>(d_dp,d_p,n,len);
        cudaDeviceSynchronize();
    }

    cudaMemcpy(dp, d_dp, sizeDP, cudaMemcpyDeviceToHost);

    cout << "Minimum cost = " << dp[1*n + (n-1)] << endl;

    cudaFree(d_dp);
    cudaFree(d_p);
    delete[] dp;
}


Writing mcm.cu


In [ ]:
!nvcc mcm.cu -o mcm
!./mcm

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Minimum cost = 30000
